# Market Anticipation Pipeline — Demo
## Stage 1: Ingest News Data → DynamoDB

In [ ]:
# !pip install boto3 requests pymysql yfinance


In [ ]:
import boto3, requests, json
from decimal import Decimal
from datetime import datetime

AWS_ACCESS_KEY_ID     = 'YOUR_AWS_ACCESS_KEY_ID'
AWS_SECRET_ACCESS_KEY = 'YOUR_AWS_SECRET_ACCESS_KEY'
REGION     = 'us-east-1'
TABLE_NAME = 'YOUR_DYNAMODB_TABLE_NAME'

dynamodb = boto3.resource(
    'dynamodb', region_name=REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
table = dynamodb.Table(TABLE_NAME)
print(f'Connected to DynamoDB table: {TABLE_NAME}')

In [ ]:
# Fetch news sentiment from Alpha Vantage
API_KEY = 'YOUR_ALPHAVANTAGE_API_KEY'
TICKER  = 'AAPL'

url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={TICKER}&apikey={API_KEY}'
r = requests.get(url)
articles = r.json().get('feed', [])
print(f'Fetched {len(articles)} articles for {TICKER}')

In [ ]:
# Preview raw structure of one article
print(json.dumps(articles[0], indent=2))

In [ ]:
# Parse and batch write to DynamoDB
DYNAMO_PK = 'YOUR_DYNAMODB_PARTITION_KEY'

def parse_article(article):
    ticker_sentiment = [
        {'ticker': t['ticker'],
         'score': Decimal(str(t['ticker_sentiment_score'])),
         'relevance': Decimal(str(t['relevance_score']))}
        for t in article.get('ticker_sentiment', [])
    ]
    topics = [t['topic'] for t in article.get('topics', [])]
    return {
        DYNAMO_PK:                 article.get('url'),           # partition key
        'time_published':          article.get('time_published'),
        'title':                   article.get('title'),
        'source':                  article.get('source'),
        'overall_sentiment_score': Decimal(str(article.get('overall_sentiment_score', 0))),
        'overall_sentiment_label': article.get('overall_sentiment_label'),
        'ticker_sentiment':        ticker_sentiment,
        'topics':                  topics,
        'summary':                 article.get('summary'),
        'ingested_at':             datetime.utcnow().isoformat()
    }

success, failed = 0, 0
with table.batch_writer() as batch:
    for article in articles:
        try:
            batch.put_item(Item=parse_article(article))
            success += 1
        except Exception as e:
            print(f'Failed: {e}')
            failed += 1

print(f'Wrote {success} articles to DynamoDB | Failed: {failed}')

In [ ]:
# Verify DynamoDB — scan 3 records
items = table.scan(Limit=3).get('Items', [])
print(f'Sample records from DynamoDB ({len(items)} shown):')
for item in items:
    print(f"\n  title:     {item['title']}")
    print(f"  published: {item['time_published']}")
    print(f"  sentiment: {item['overall_sentiment_label']} ({item['overall_sentiment_score']})")
    print(f"  source:    {item['source']}")

## Stage 2: Ingest S&P 500 Stock Price → Aurora (MySQL)

In [ ]:
import pymysql, json

# Retrieve Aurora password from Secrets Manager
SECRET_ARN = 'YOUR_SECRETS_MANAGER_ARN'

sm = boto3.client('secretsmanager', region_name='us-east-1',
                  aws_access_key_id=AWS_ACCESS_KEY_ID,
                  aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
secret = sm.get_secret_value(SecretId=SECRET_ARN)
DB_PASSWORD = json.loads(secret['SecretString'])['password']

DB_HOST = 'YOUR_AURORA_ENDPOINT'
DB_USER = 'YOUR_AURORA_USERNAME'
DB_PORT = 3306
SSL_CA  = './global-bundle.pem'

conn = pymysql.connect(
    host=DB_HOST, port=DB_PORT,
    user=DB_USER, password=DB_PASSWORD,
    ssl={'ca': SSL_CA}
)
print(f'Connected to Aurora MySQL {conn.get_server_info()}')

In [ ]:
# Create database and stock_prices table
with conn.cursor() as cur:
    cur.execute('CREATE DATABASE IF NOT EXISTS market_data')
    cur.execute('USE market_data')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS stock_prices (
            id     BIGINT AUTO_INCREMENT PRIMARY KEY,
            ticker VARCHAR(10)  NOT NULL,
            ts     DATETIME     NOT NULL,
            open   DECIMAL(12,4),
            high   DECIMAL(12,4),
            low    DECIMAL(12,4),
            close  DECIMAL(12,4),
            volume BIGINT,
            UNIQUE KEY uq_ticker_ts (ticker, ts)
        )
    ''')
    conn.commit()
print("Table 'market_data.stock_prices' ready.")

In [ ]:
import yfinance as yf

# Pull S&P 500 daily data — last 10 years
sp500 = yf.download('^GSPC', period='10y', interval='1d', auto_adjust=True)
sp500.columns = [c[0].lower() for c in sp500.columns]
sp500 = sp500[['open','high','low','close','volume']].dropna()
sp500.index = sp500.index.tz_localize(None)

print(f'Downloaded {len(sp500)} rows  ({sp500.index[0].date()} to {sp500.index[-1].date()})')
sp500.head()

In [ ]:
# Batch insert into Aurora
sql = '''
    INSERT INTO stock_prices (ticker, ts, open, high, low, close, volume)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        open=VALUES(open), high=VALUES(high), low=VALUES(low),
        close=VALUES(close), volume=VALUES(volume)
'''
rows = [
    ('^GSPC', ts.to_pydatetime(),
     float(row.open), float(row.high), float(row.low),
     float(row.close), int(row.volume))
    for ts, row in sp500.iterrows()
]

with conn.cursor() as cur:
    cur.execute('USE market_data')
    for i in range(0, len(rows), 500):
        cur.executemany(sql, rows[i:i+500])
    conn.commit()

print(f'Inserted {len(rows)} rows into Aurora stock_prices.')

In [ ]:
# Verify Aurora — query latest 5 rows
with conn.cursor() as cur:
    cur.execute('USE market_data')
    cur.execute('SELECT ticker, ts, open, high, low, close, volume FROM stock_prices ORDER BY ts DESC LIMIT 5')
    results = cur.fetchall()

print(f"{'ticker':<8} {'ts':<22} {'open':>10} {'high':>10} {'low':>10} {'close':>10} {'volume':>14}")
print('-' * 90)
for r in results:
    print(f"{r[0]:<8} {str(r[1]):<22} {float(r[2]):>10.2f} {float(r[3]):>10.2f} {float(r[4]):>10.2f} {float(r[5]):>10.2f} {r[6]:>14,}")

conn.close()
print('\nConnection closed.')